# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, following a reproducible workflow for discovering and analyzing the data.

### Dataset Source
The dataset metadata and schema are published as a Croissant file, enabling structured exploration, and are accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will initialize the dataset object and retrieve high-level metadata such as the dataset's name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Fetch dataset-level metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` entries. We'll enumerate record sets and then explore which fields and columns are included in each record set.

To ensure reproducibility and consistent referencing, **all entities are referenced by their `@id`**.

Let's list the record sets in the schema, view their `@id`s, then for each, enumerate the field and column `@id`s.

In [ ]:
# List all record sets available, with their @id and fields
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"[RecordSet] @id: {rs.id}")
    # List fields for this record set
    print("  Fields (@id):")
    for field in rs.fields:
        colid = getattr(field.column, 'id', None)
        print(f"   - {field.id} (column: {colid})")
    print("  Source columns:")
    for c in rs.columns:
        print(f"   - {c.id}")
    print()

# For reference below, gather IDs for exploration
record_set_ids = [rs.id for rs in record_sets]
field_id_examples = {}
for rs in record_sets:
    if rs.fields:
        # Save first field for demonstration
        field_id_examples[rs.id] = rs.fields[0].id

## 3. Data Extraction
Let's load each record set's data into a Pandas DataFrame for analysis. We'll use the `@id` of each record set.

We'll print out the columns in the first available record set as an example, and display the top 5 records.

In [ ]:
# Extracting all record sets into DataFrames by @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for RecordSet: {rs_id}")
        else:
            print(f"No records found for RecordSet: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Select first populated DataFrame for demonstration
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in first available record set ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from one of the main record sets (for example, 'coverage_percent', 'peptide_count', or 'MW'), filter records where the value is above a given threshold, normalize the column, and (if available) group the data by another field (for example, 'accession' or 'description'). All references use `@id`.

> _Replace `cr:Field:coverage_percent`, `cr:Field:peptide_count`, etc., with the actual `@id` of the relevant numeric field(s) as shown previously._ For this dataset, typical Croissant field identifiers are often of the form `cr:Field:<field_name>`.

We'll attempt to automatically select a numeric field from the DataFrame for the example workflow.

In [ ]:
# Automatically select a numeric field from columns
from pandas.api.types import is_numeric_dtype

record_set_id = first_rs_id  # Use the first non-empty record set
df = dataframes[record_set_id]
# Display column names for reference
print(f"Columns for {record_set_id}: {df.columns.tolist()}")

# Select numeric fields by dtype
numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    print("No numeric field detected. Skipping EDA section.")

# Filter by a threshold (e.g., value > 10)
if numeric_fields:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a string field (if available)
    string_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if string_fields:
        group_field = string_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (showing average {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable string field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the numeric field and, if grouped, provide a bar plot of group means.

This is a basic overview; feel free to tailor further based on available fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Grouped bar plot if grouping was performed
    if 'grouped_df' in locals():
        grouped_df = grouped_df.sort_values(by=numeric_field, ascending=False)[:20]  # Show top 20
        plt.figure(figsize=(8, 6))
        sns.barplot(x=grouped_df[numeric_field], y=grouped_df.index)
        plt.title(f"Top 20 groups by mean {numeric_field} (grouped by {group_field})")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
We loaded the FAIR² mass spectrometry dataset from Croissant metadata, surveyed available record sets and fields by `@id`, and analyzed selected numeric data fields. Further advanced analyses can be conducted using these normalized and grouped data, depending on specific research questions.

*Key learnings:*
- Croissant schemas enable programmatic access, universal referencing, and transparent EDA workflows.
- Referencing all dataset entities by their `@id` ensures reproducibility and consistency.
- A combination of `mlcroissant`, Pandas, and standard EDA tools makes structured interactive exploration possible for complex scientific datasets.